In [1]:
# Install dependencies
%pip install -U torch torchvision segmentation-models-pytorch albumentations opencv-python matplotlib pandas numpy kagglehub

In [2]:
%pip install numpy==1.24.3 scipy==1.11.4 scikit-image==0.21.0
%pip install --force-reinstall albumentations

  Using cached numpy-1.24.3.tar.gz (10.9 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadat

In [3]:
%pip install --upgrade --force-reinstall albumentations

  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached stringzilla-4.6.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_28_x86_64.whl.metadata (121 kB)
  Using cached simsimd-6.5.16-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (70 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2

In [4]:
import os
import cv2
import copy
import torch
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader, Subset
import kagglehub

# Reproducibilityz

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# Konfigurasi Pelatihan
CONFIG = {
    "IMAGE_WIDTH": 512,
    "IMAGE_HEIGHT": 384,
    "BATCH_SIZE": 8,
    "EPOCHS": 60,
    "LR": 1e-4,
    "NUM_CLASSES": 1,
    "VAL_RATIO": 0.2,
    "RANDOM_SEED": 42,
    "PATIENCE": 8,
    "MIN_LR": 1e-6,
    "THRESHOLD_CANDIDATES": [round(x, 2) for x in np.arange(0.30, 0.71, 0.05)],
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu"
}
seed_everything(CONFIG["RANDOM_SEED"])
print(f"Menggunakan device: {CONFIG['DEVICE']}")

# 1. Unduh dataset FAD menggunakan kagglehub
path = kagglehub.dataset_download("faizalkarim/flood-area-segmentation")
print(f"Dataset tersimpan di: {path}")

# 2. Setup path
csv_path = os.path.join(path, "metadata.csv")
image_dir = os.path.join(path, "Image")
mask_dir = os.path.join(path, "Mask")

# 3. Class Dataset
class FADDataset(Dataset):
    def __init__(self, csv_file, img_dir, mask_dir, transform=None):
        self.data_info = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_info)

    def __getitem__(self, idx):
        img_name = self.data_info.iloc[idx, 0]
        mask_name = self.data_info.iloc[idx, 1]

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        mask = mask.unsqueeze(0)
        return image, mask

# 4. Augmentasi & DataLoader
train_transform = A.Compose([
    A.Resize(CONFIG["IMAGE_HEIGHT"], CONFIG["IMAGE_WIDTH"]),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(CONFIG["IMAGE_HEIGHT"], CONFIG["IMAGE_WIDTH"]),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

metadata = pd.read_csv(csv_path)
num_samples = len(metadata)
indices = torch.randperm(num_samples, generator=torch.Generator().manual_seed(CONFIG["RANDOM_SEED"])).tolist()
val_size = int(num_samples * CONFIG["VAL_RATIO"])
val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_dataset = Subset(FADDataset(csv_path, image_dir, mask_dir, transform=train_transform), train_indices)
val_dataset = Subset(FADDataset(csv_path, image_dir, mask_dir, transform=val_transform), val_indices)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=False, num_workers=0, pin_memory=True)

print(f"Total gambar: {num_samples} | Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Menggunakan device: cpu


100%|██████████| 107M/107M [00:04<00:00, 22.8MB/s] 

Extracting files...


Dataset tersimpan di: /root/.cache/kagglehub/datasets/faizalkarim/flood-area-segmentation/versions/1
Total gambar: 290 | Train: 232 | Val: 58


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
# Inisialisasi U-Net dengan backbone tangguh
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    in_channels=3,
    classes=CONFIG["NUM_CLASSES"],
)
model = model.to(CONFIG["DEVICE"])

# Loss gabungan untuk menangani overlap + class imbalance
criterion_dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
criterion_bce = nn.BCEWithLogitsLoss()

def criterion(logits, masks):
    return 0.5 * criterion_dice(logits, masks) + 0.5 * criterion_bce(logits, masks)

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["LR"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2,
    min_lr=CONFIG["MIN_LR"]
)
scaler = torch.cuda.amp.GradScaler(enabled=(CONFIG["DEVICE"] == "cuda"))

# Fungsi pembantu

def calculate_iou(pred_logits, target, threshold=0.5):
    pred = (pred_logits.sigmoid() > threshold).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    if union == 0:
        return 1.0
    return (intersection / union).item()


def evaluate_model(loader, threshold=0.5):
    model.eval()
    total_loss = 0.0
    total_iou = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(CONFIG["DEVICE"], non_blocking=True)
            masks = masks.to(CONFIG["DEVICE"], non_blocking=True)

            logits = model(images)
            loss = criterion(logits, masks)

            total_loss += loss.item()
            total_iou += calculate_iou(logits, masks, threshold=threshold)

    return total_loss / max(len(loader), 1), total_iou / max(len(loader), 1)


def find_best_threshold(loader, candidates):
    best_threshold = 0.5
    best_iou = -1.0

    for threshold in candidates:
        _, iou = evaluate_model(loader, threshold=threshold)
        if iou > best_iou:
            best_iou = iou
            best_threshold = threshold

    return best_threshold, best_iou


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

/tmp/ipykernel_6601/4095929374.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(CONFIG["DEVICE"] == "cuda"))


In [ ]:
print("Memulai Pelatihan...")

best_val_iou = -1.0
best_epoch = -1
patience_counter = 0
best_state_dict = copy.deepcopy(model.state_dict())

for epoch in range(CONFIG["EPOCHS"]):
    model.train()
    running_loss = 0.0
    running_iou = 0.0

    for images, masks in train_loader:
        images = images.to(CONFIG["DEVICE"], non_blocking=True)
        masks = masks.to(CONFIG["DEVICE"], non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(CONFIG["DEVICE"] == "cuda")):
            outputs = model(images)
            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_iou += calculate_iou(outputs.detach(), masks, threshold=0.5)

    train_loss = running_loss / max(len(train_loader), 1)
    train_iou = running_iou / max(len(train_loader), 1)

    val_loss, val_iou = evaluate_model(val_loader, threshold=0.5)
    scheduler.step(val_iou)

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch [{epoch+1}/{CONFIG['EPOCHS']}] | "
        f"Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f} | LR: {current_lr:.2e}"
    )

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_epoch = epoch + 1
        patience_counter = 0
        best_state_dict = copy.deepcopy(model.state_dict())
    else:
        patience_counter += 1

    if patience_counter >= CONFIG["PATIENCE"]:
        print(f"Early stopping aktif di epoch {epoch+1}.")
        break

model.load_state_dict(best_state_dict)
print(f"Model terbaik di epoch {best_epoch} dengan Val IoU: {best_val_iou:.4f}")

best_threshold, best_threshold_iou = find_best_threshold(val_loader, CONFIG["THRESHOLD_CANDIDATES"])
print(f"Best threshold (berdasarkan val): {best_threshold:.2f} | IoU: {best_threshold_iou:.4f}")
print("Pelatihan Selesai!")

Memulai Pelatihan...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_6601/185783566.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(CONFIG["DEVICE"] == "cuda")):


KeyboardInterrupt: 

In [ ]:
# Ambil satu batch data validasi untuk visualisasi
model.eval()
images, masks = next(iter(val_loader))
images = images.to(CONFIG["DEVICE"])

with torch.no_grad():
    preds = model(images)
    preds = (preds.sigmoid() > best_threshold).float().cpu()

# Visualisasikan gambar pertama dari batch
img_show = images[0].cpu().numpy().transpose(1, 2, 0)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_show = std * img_show + mean
img_show = np.clip(img_show, 0, 1)

mask_show = masks[0].cpu().numpy().squeeze()
pred_show = preds[0].numpy().squeeze()

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(img_show)
ax[0].set_title("Gambar Asli (Val)")
ax[0].axis("off")

ax[1].imshow(mask_show, cmap="gray")
ax[1].set_title("Ground Truth (Val)")
ax[1].axis("off")

ax[2].imshow(img_show)
ax[2].imshow(pred_show, cmap="jet", alpha=0.4)
ax[2].set_title(f"Prediksi Model (thr={best_threshold:.2f})")
ax[2].axis("off")

plt.show()


In [ ]:
model.eval()

# Global metrics di validation set
total_intersection = 0.0
total_union = 0.0
total_tp = 0.0
total_fp = 0.0
total_fn = 0.0

print("Evaluating model on validation set...")

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(CONFIG['DEVICE'])
        masks = masks.to(CONFIG['DEVICE'])

        outputs = model(images)
        preds = (outputs.sigmoid() > best_threshold).float()

        preds_flat = preds.view(-1)
        masks_flat = masks.view(-1)

        intersection = (preds_flat * masks_flat).sum().item()
        union = preds_flat.sum().item() + masks_flat.sum().item() - intersection
        total_intersection += intersection
        total_union += union

        total_tp += intersection
        total_fp += (preds_flat * (1 - masks_flat)).sum().item()
        total_fn += ((1 - preds_flat) * masks_flat).sum().item()

mean_iou = total_intersection / (total_union + 1e-8)
mean_precision = total_tp / (total_tp + total_fp + 1e-8)
mean_recall = total_tp / (total_tp + total_fn + 1e-8)
f1_score = 2 * (mean_precision * mean_recall) / (mean_precision + mean_recall + 1e-8)

print("--- Validation Results ---")
print(f"Threshold: {best_threshold:.2f}")
print(f"Mean IoU: {mean_iou:.4f}")
print(f"Mean Precision: {mean_precision:.4f}")
print(f"Mean Recall: {mean_recall:.4f}")
print(f"F1-Score: {f1_score:.4f}")
